Load

In [8]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import tensorflow as tf
import joblib

from src.modeling import (
    train_test_split_time_series,
    scale_data,
    create_sequences_return,
    build_lstm_return,
    evaluate_model,
    add_log_return
)

df = pd.read_csv("../data/processed/dataset_final.csv")
df = df.sort_values("date")

df = add_log_return(df)

df_model = df[["ihsg_return"]]

Split

In [9]:
train, test = train_test_split_time_series(df_model)

Scaling

In [10]:
train_scaled, test_scaled, scaler = scale_data(train, test)

Windowing

In [11]:
window_size = 60

X_train, y_train = create_sequences_return(train_scaled, window_size)
X_test, y_test = create_sequences_return(test_scaled, window_size)

test

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [13]:
def build_lstm_return_tuned(input_shape, units1=64, units2=32, lr=0.001):
    model = tf.keras.Sequential([
        tf.keras.layers.LSTM(units1, return_sequences=True, input_shape=input_shape),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(units2),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1)
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss="mse"
    )

    return model

In [14]:
results = []

window_options = [20, 30, 60]
unit_options = [(64,32), (128,64)]
lr_options = [0.001, 0.0005]

for window_size in window_options:

    X_train, y_train = create_sequences_return(train_scaled, window_size)
    X_test, y_test = create_sequences_return(test_scaled, window_size)

    for units in unit_options:
        for lr in lr_options:

            print(f"\nWindow={window_size}, Units={units}, LR={lr}")

            model = build_lstm_return_tuned(
                (X_train.shape[1], X_train.shape[2]),
                units1=units[0],
                units2=units[1],
                lr=lr
            )

            history = model.fit(
                X_train,
                y_train,
                epochs=100,
                batch_size=32,
                validation_split=0.1,
                callbacks=[early_stop],
                verbose=0
            )

            y_pred = model.predict(X_test, verbose=0)

            rmse, mape, r2 = evaluate_model(y_test, y_pred)

            results.append({
                "window": window_size,
                "units": units,
                "lr": lr,
                "rmse": rmse,
                "mape": mape,
                "r2": r2
            })

            print("RMSE:", rmse)
            print("R2:", r2)


Window=20, Units=(64, 32), LR=0.001


d:\project_skripsi\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE: 0.04347025467499047
R2: -0.0010002673747913082

Window=20, Units=(64, 32), LR=0.0005


d:\project_skripsi\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE: 0.04606886246525265
R2: -0.12425493314166292

Window=20, Units=(128, 64), LR=0.001


d:\project_skripsi\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


... (15 outputs left out)

d:\project_skripsi\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE: 0.045483180520983345
R2: -0.06030289872724226

Window=60, Units=(128, 64), LR=0.0005


d:\project_skripsi\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


RMSE: 0.04526699621586576
R2: -0.05024748516590627


In [15]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df.sort_values("rmse").head(10)

,window,units,lr,rmse,mape,r2
0,20,"(64, 32)",0.0010,0.043470,0.084571,-0.001000
2,20,"(128, 64)",0.0010,0.044364,0.085959,-0.042598
6,30,"(128, 64)",0.0010,0.044443,0.085037,-0.035351
7,30,"(128, 64)",0.0005,0.044635,0.085239,-0.044313
9,60,"(64, 32)",0.0005,0.044927,0.086849,-0.034544
11,60,"(128, 64)",0.0005,0.045267,0.087819,-0.050247
8,60,"(64, 32)",0.0010,0.045357,0.086635,-0.054424
10,60,"(128, 64)",0.0010,0.045483,0.088934,-0.060303
4,30,"(64, 32)",0.0010,0.045752,0.086972,-0.097233
1,20,"(64, 32)",0.0005,0.046069,0.091159,-0.124255


Build Model

In [6]:
model = build_lstm_return((X_train.shape[1], X_train.shape[2]))

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

d:\project_skripsi\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - loss: 0.0116 - val_loss: 0.0027
Epoch 2/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.0040 - val_loss: 0.0027
Epoch 3/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - loss: 0.0040 - val_loss: 0.0027
Epoch 4/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.0040 - val_loss: 0.0030
Epoch 5/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.0042 - val_loss: 0.0031
Epoch 6/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - loss: 0.0041 - val_loss: 0.0027
Epoch 7/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - loss: 0.0040 - val_loss: 0.0028
Epoch 8/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.0040 - val_loss: 0.0027
Epoch 9/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.0040 - val_loss: 0.0030
Epoch 10/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 0.0040 - val_loss: 0.0027
Epoch 11/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.0040 - val_loss: 0.0027
Epoch 12/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.0

Evaluasi

In [7]:
y_pred = model.predict(X_test)

rmse, mape, r2 = evaluate_model(y_test, y_pred)

print("RMSE:", rmse)
print("MAPE:", mape)
print("R2:", r2)

15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step
RMSE: 0.04385538700257158
MAPE: 0.08475957490327432
R2: -0.008146203685402975
